In [55]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')
import torch, gc
# gc.collect()
# torch.cuda.empty_cache() # 👈 强制清空显存碎片
#=========================================================
# os.environ['TF_FORCE_GPU_ALLOW_GROWTH']='true'

                         
#=======================================================
!pip install webdataset livelossplot
import webdataset as wds
import json
import re
import torch.nn as nn
import pandas as pd
import tiktoken
import torch
import os
import glob
from datetime import datetime
# os.environ['TF_ENABLE_ONEDNN_OPTS'] = '1'
# 只让程序看到第 1 号显卡（即物理上的第 2 张卡）。此时程序里的 cuda:0 就是物理上的 cuda:1
# os.environ["CUDA_VISIBLE_DEVICES"] = "1" 

import torch
# 此时 torch 只能看到一张卡，默认就是你要的 cuda:1
from torch.utils.data import DataLoader, Dataset, IterableDataset, random_split
import logging
from rich.logging import RichHandler
from livelossplot import PlotLosses
from torch.utils.tensorboard import SummaryWriter
from transformers import AutoTokenizer

# import torch.distributed as dist
# import torch.multiprocessing as mp
# from torch.nn.parallel import DistributedDataParallel as DDP
# from torch.utils.data import DataLoader, DistributedSampler
# tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased", use_fast=True)

# from mainScore import format_input, check_if_running, generate_model_scores, query_model, console
from rich.console import Console
console = Console()
# os.environ["MASTER_ADDR"] = "127.0.0.1"
# os.environ["MASTER_PORT"] = "12355"
# dist.init_process_group("nccl", rank=rank, world_size=world_size)
# device = torch.device(f"cuda:{rank}")
# torch.cuda.set_device(rank)
############################ MODEL CONFIG ##########################
GPT_CONFIG_124M = {
    "model": "gpt2",
    "context_length": 1024,
    "n_layers": 12,
    "n_heads": 12,
    "emb_dim": 768,
    "drop_rate": 0.1,
    "temperature": 0.7,
    "vocab_size": 50260,
    "qkv_bias": True
}
MODEL_CONFIGS = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}
model_name = "gpt2-small (124M)"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
base_enc = tiktoken.get_encoding("gpt2")
# new_special_tokens = {"additional_special_tokens": ["<|persona|>", "<|user|>", "<|bot|>"]}
new_special_tokens = {
    "<|persona|>": 50257,
    "<|user|>": 50258,
    "<|bot|>": 50259
}
# tokenizer.add_special_tokens(special_tokens)
tokenizer = tiktoken.Encoding(
    name="gpt2-custom",
    pat_str=base_enc._pat_str,
    mergeable_ranks=base_enc._mergeable_ranks,
    special_tokens={
        **base_enc._special_tokens,
        **new_special_tokens
    }
)

# 3. 调整模型嵌入层大小，容纳新 token
NEW_CONFIG = GPT_CONFIG_124M.copy()
NEW_CONFIG.update(MODEL_CONFIGS[model_name])
############################ Tokenizer ##########################
# class SimpleTokenizer():
#     def __init__(self, vocab):
#         # 初始化词汇表, 编码 str to int
#         self.str_to_int = vocab
#         # 初始化词汇表, 解码 int to str
#         self.int_to_str = {i:s for s, i in vocab.items()}
#     # 编码文本， 返回 int 序列
#     def encode(self, text):
#         # 对文本进行预处理，将标点符号和空格作为单独的 token
#         preprocessed = re.split(r'([,.?_!"()\']|--|\s)', text)
#         preprocessed = [
#             # 对每个 token 进行预处理，将空格替换为下划线
#             item.strip() for item in preprocessed if item.strip()
#         ]
#         preprocessed = [item if item in self.str_to_int
#                         else "<|unk|>" for item in preprocessed]
#         # 对每个 token 进行编码
#         ids = [self.str_to_int[s] for s in preprocessed]
#         # 返回编码后的 int 序列
#         return ids
#     # 解码 int 序列， 返回文本字符串
#     def decode(self, ids):
#         # 对每个 id 进行解码
#         text = "".join([self.int_to_str[i] for i in ids])
#         # 对文本进行后处理，将下划线替换为空格
#         text = re.sub(r'\s+([,.?_!"()\'])', r'\1', text)
#         # 返回解码后的文本字符串
#         return text
# ############################ Tokenizer Gpt2##########################
# print(tokenizer.encode("<|bot|>", allowed_special={"<|bot|>"}))

/kaggle/input/datasets/mangzhang/the-verdict/the-verdict.txt
/kaggle/input/datasets/mangzhang/sf-data/sft_data.json
/kaggle/input/datasets/mangzhang/cnliteralconversation/allChineseLiteral_sft_custom.json
/kaggle/input/datasets/mangzhang/allchineseliteralcn/allChineseLiteralCN.pth
/kaggle/input/datasets/mangzhang/qwennewtrainclean/pretrain_body_clean.txt
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 649.8/649.8 kB 4.4 MB/s  0:00:00
  Attempting uninstall: psutil
    Found existing installation: psutil 5.9.5
    Uninstalling psutil-5.9.5:
      Successfully uninstalled psutil-5.9.5━━━━━━━━━━━━━━━━━━━━━━━ 3/8 [psutil]
  Attempting uninstall: ipython━━╺━━━━━━━━━━━━━━ 5/8 [stack_data]
    Found existing installation: ipython 7.34.0m━━━━━━━━━━━━━━ 5/8 [stack_data]
    Uninstalling ipython-7.34.0:m╺━━━━━━━━━━━━━━ 5/8 [stack_data]
      Successfully uninstalled ipython-7.34.00m╺━━━━━━━━━ 6/8 [ipython]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [livelossplot] [ipython]
ERROR: pip's dependen

I0000 00:00:1795045328.826282      50 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1795045343.089266      50 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [56]:
# 如果你是在计算 API 费用或对接 ChatGPT，千万别用 GPT-2 的编码，必须用 tiktoken.get_encoding("cl100k_base")（对于更新的 GPT-4o 则是 o200k_base）。词表也变大了（cl100k_base 通常约 10 万级别）。
############################ FeedForward ############################
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            # 768 -> 3072
            nn.GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"])
            # 3072 -> 768
        )
    def forward(self, x):
        return self.layers(x)

In [57]:
############################ LayerNorm ##############################
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))
        
    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

In [58]:
############################ MultiHeadAttention ######################
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, num_heads, dropout, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                    diagonal=1)
        )
    def forward(self, x, attention_mask=None):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        values = self.W_value(x)
        queries = self.W_query(x)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(
            b, num_tokens, self.num_heads, self.head_dim
        )
        keys = keys.transpose(1,2)
        queries = queries.transpose(1,2)
        values = values.transpose(1,2)
        
        attn_scores = queries @ keys.transpose(2,3)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens] # type: ignore
        
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        if attention_mask is not None:
            extended_mask = attention_mask[:,None, None,:]
            attn_scores = attn_scores.masked_fill(extended_mask == 0, -torch.inf)
        
        attn_weights = torch.softmax(
            attn_scores /keys.shape[-1]**0.5, dim=-1
        )
        context_vec = (attn_weights @ values).transpose(1,2)
        context_vec = context_vec.contiguous().view(
            b, num_tokens, self.d_out
        )
        context_vec = self.out_proj(context_vec)
        return context_vec        
        
############################ CausalAttention ######################

In [59]:
############################ Transformer Block ######################
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
        d_in = cfg["emb_dim"],
        d_out = cfg["emb_dim"],
        context_length = cfg["context_length"],
        num_heads = cfg["n_heads"],
        dropout = cfg["drop_rate"],
        qkv_bias = cfg["qkv_bias"]
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])
    def forward(self, x, attention_mask=None):
        # 第一个子层：多头因果注意力
        shortcut = x            # 1. 备份原始输入
        x = self.norm1(x)       # 2. 层归一化（稳定数值）
        x = self.att(x, attention_mask)         # 3. 多头因果注意力（提取上下文关系）
        x = self.drop_shortcut(x) # 4. Dropout（训练时防止过拟合，推理时关闭）
        x = x + shortcut        # 5. 第一次残差连接（将处理后的特征与原始输入相加）
        
        # 第二个子层：前馈神经网络
        shortcut = x             # 🚨 注意：这里拼写错了！应该是 shortcut
        x = self.norm2(x)       # 6. 再次层归一化
        x = self.ff(x)          # 7. 前馈网络（先膨胀到 4 倍，GELU 激活，再收缩回原维度）
        x = self.drop_shortcut(x) # 8. Dropout
        x = x + shortcut        # 9. 第二次残差连接
        return x

In [60]:
############################ GPT MODEL ##########################
class GPT_MODEL(torch.nn.Module):
    
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        self.trf_blocks = nn.ModuleList(
            [TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
            
        )
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )
    def forward(self, input_ids, attention_mask=None, labels=None):
        batch_size, seq_len = input_ids.shape
    
        tok_embeds = self.tok_emb(input_ids)
        pos_embeds = self.pos_emb(
            torch.arange(seq_len, device=input_ids.device)
        )
    
        x = self.drop_emb(tok_embeds + pos_embeds)
    
        for block in self.trf_blocks:
            x = block(x, attention_mask)
    
        x = self.final_norm(x)
        logits = self.out_head(x)
    
        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
    
            valid_targets = shift_labels != -100
            if not valid_targets.any():
                raise ValueError(
                    "labels 全部为 -100；请检查 <|bot|> token ID 和 collate_fn"
                )
    
            loss = torch.nn.functional.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=-100,
            )
    
        return {"logits": logits, "loss": loss}



        
    # def forward(self,  input_ids, attention_mask=None, labels=None):
    #     batch_size, seq_len= input_ids.shape
    #     tok_embeds = self.tok_emb(input_ids)
    #     pos_embeds = self.pos_emb(
    #         torch.arange(seq_len, device=input_ids.device)
    #     )
    #     x = tok_embeds + pos_embeds
    #     x = self.drop_emb(x)
    #     for block in self.trf_blocks:
    #         x = block(x, attention_mask) ######书上没有，必须
    #     x = self.final_norm(x)
    #     logits = self.out_head(x)
        
    #     loss = None
    #     if labels is not None:
    #         loss = torch.nn.functional.cross_entropy(
    #             logits.view(-1, logits.size(-1)),
    #             labels.view(-1),
    #             ignore_index=-100
    #         )
        
        
    #     return {"logits": logits, "loss": loss}
############################ MODEL ############################

In [61]:
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(txt)
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length +1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))
    def __len__(self):
        return len(self.input_ids)
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [62]:
def create_dataloader(txt, batch_size=2, max_length=256,
                     stride=128, shuffle=True, drop_last=True,
                     num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        drop_last=drop_last,
        num_workers=num_workers
    )
    return dataloader

In [63]:
with open("/kaggle/input/datasets/mangzhang/the-verdict/the-verdict.txt") as f:
    txt = f.read()
dataloader = create_dataloader(
    txt, batch_size=2, max_length=1024, stride=1024, shuffle=False
)
data_iter = iter(dataloader)
first_batch = next(data_iter)

In [64]:
############################ WebDataset ############################
def create_text_webdataset(file_path, output_dir, chunk_size = 2000,max_shards_size=10000):
    """
    把文本文件 转换成 WebDataset 的 .tar 分片。
    
    :param file_path: 包含文本数据的文件路径
    :param output_dir: 输出路径，例如 "shards/retail-%06d.tar"
    :param max_shards_size: 每个分片包含的样本数
    """
    # SharedWriter 会自动按maxcount分片
    os.makedirs(output_dir, exist_ok=True)
    output_pattern = os.path.join(output_dir, "literal-%06d.tar")
    sample_idx =0
    with open(file_path, "r", encoding="utf-8") as f:
        raw_text = f.read()
    # raw_text = "".join(raw_text)
    total_len = len(raw_text)
    print(f"total text len: {total_len}")
    split_pattern = re.compile(r'([。!?])')
    sentences = split_pattern.split(raw_text)
    pieces = []
    buf = ""
    for seg in sentences:
        buf += seg
        if len(buf) >= chunk_size:
            pieces.append(buf)
            buf = ""
    if len(buf.strip()) > 0:
        pieces.append(buf)
        
    with wds.ShardWriter(output_pattern, maxcount=max_shards_size) as sink: # type: ignore
        # 遍历 DataFrame 中的每一行
        for block in pieces:
            if len(block) < 50:
                continue
            sample = {
                "__key__": f"chunk_{sample_idx:08d}",
                "text": block.encode("utf-8") # type: ignore
            }
            sink.write(sample)
            sample_idx += 1
    print(f"Total samples blocks: {sample_idx}")
            # console.print(sample.values())

In [65]:
############################Prepare Data############################
def decode_txt(sample):
    """
    自定义解码函数：当样本里包含 'txt' 字段时，将字节流解码为字符串。
    """
    if "text" in sample:
        sample["text"] = sample["text"].decode("utf-8")
    return sample
def prepare_data():
    file_path = "D:\\Project\\AI\\ZeroToTransformer\\allchineseliteral_clean.txt"
    # 生成分片，每个分片 10000 条样本
    create_text_webdataset(file_path,  "shardsTxt", max_shards_size=10000)
############################ MODEL#################################

model = GPT_MODEL(NEW_CONFIG)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.eval()
model.to(device)
# model = torch.nn.DataParallel(model, device_ids=[0, 1]).to(device)
############################Train data ############################
# 1. 定义一个专门读取散装 text 文件的 Dataset
class KaggleTextDataset(Dataset):
    def __init__(self, folder_path):
        # 自动找到文件夹下所有以 .text 结尾的文件
        folder_path = "/kaggle/input/datasets/mangzhang/the-verdict/"
        self.file_paths = glob.glob(os.path.join(folder_path, "*.txt"))
        if not self.file_paths:
            raise FileNotFoundError(f"在 {folder_path} 下没有找到任何 .text 文件，请检查路径！")
        print(f"✅ 成功找到 {len(self.file_paths)} 个文本文件")
        
    def __len__(self):
        return len(self.file_paths)
        
    def __getitem__(self, idx):
        # 每次读取一个文件
        with open(self.file_paths[idx], "r", encoding="utf-8") as f:
            text = f.read()
        # 返回一个元组，和之前 WebDataset 的 .to_tuple("text") 格式保持一致
        return (text,)
def collate_fn(batch):
    texts = [item[0].strip() for item in batch]
    encoded_texts = [tokenizer.encode(text, allowed_special={"<|persona|>", "<|user|>", "<|bot|>"})[:1024] for text in texts]
    max_len = max(len(ids) for ids in encoded_texts)
    pad_token_id = 50256
    padded_ids = []
    attention_masks = []
    for ids in encoded_texts:
        padded = ids + [pad_token_id] * (max_len -len(ids))
        mask = [1] * len(ids) + [0] * (max_len -len(ids))
        padded_ids.append(padded)
        attention_masks.append(mask)
    input_ids = torch.tensor(padded_ids, dtype=torch.long)
    attention_masks = torch.tensor(attention_masks, dtype=torch.long)
    labels = input_ids.clone()
    labels[labels == pad_token_id] = -100
    labels = torch.cat([labels[:, 1:],
                    torch.full((labels.size(0), 1), -100, dtype=torch.long)], dim=1)
    return {
        "input_ids": input_ids,
        "attention_mask": attention_masks,
        "labels": labels,
    }

In [66]:
import torch
from torch.utils.data import Dataset, DataLoader

class SingleFileTextDataset(Dataset):
    def __init__(self, file_path, chunk_size=1000):
        with open(file_path, "r", encoding="utf-8") as f:
            raw_text = f.read()
        # 切分成一个个 chunk 列表
        self.chunks = [raw_text[i:i+chunk_size] for i in range(0, len(raw_text), chunk_size)]
        # 过滤掉太短的
        # self.chunks = [c for c in self.chunks if len(c.strip()) > 10]
        
    def __len__(self):
        return len(self.chunks)
        
    def __getitem__(self, idx):
        return (self.chunks[idx],)

#######################################
class SingleFileTextDataset2(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(txt)
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length +1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))
    def __len__(self):
        return len(self.input_ids)
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]





###############################################
# 1. 实例化整个 Dataset
full_dataset = SingleFileTextDataset("/kaggle/input/datasets/mangzhang/qwennewtrainclean/pretrain_body_clean.txt")
total_size = len(full_dataset) # 比如 2918

# 2. 计算切分点
train_size = int(total_size * 0.8)
val_size = int(total_size * 0.1)
# test_size 就是剩下的 10%

# 3. 你完全可以像切列表一样切 Dataset！（因为底层数据是 List）
train_dataset = torch.utils.data.Subset(full_dataset, range(0, train_size))
val_dataset = torch.utils.data.Subset(full_dataset, range(train_size, train_size + val_size))
test_dataset = torch.utils.data.Subset(full_dataset, range(train_size + val_size, total_size))

# 4. 直接扔进 DataLoader
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, collate_fn=collate_fn)

In [67]:
# tiny = SFTDataset(cleaned[:1])
# tiny_loader = DataLoader(tiny, batch_size=1, shuffle=True, collate_fn=collate_fn)

# for n, p in model.named_parameters():
#     p.requires_grad_("lora" in n.lower())
# opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=5e-4)

# model.to(device); model.train()
# for i in range(200):
#     for bb in tiny_loader:
#         opt.zero_grad()
#         l = model(input_ids=bb["input_ids"].to(device),
#                   attention_mask=bb["attention_mask"].to(device),
#                   labels=bb["labels"].to(device))["loss"]
#         l.backward(); opt.step()
#     if i % 20 == 0:
#         print(i, round(l.item(), 4))

# # 立刻背诵测试
# model.eval()
# s0 = cleaned[0]
# print(generate_text(model, build_prompt(s0["system"], s0["instruction"]),
#                     max_new_tokens=300, greedy=True))
# print("---标准答案---")
# print(s0["output"][:300])


In [68]:
from torch.optim.lr_scheduler import CosineAnnealingLR

 
def calc_loss_batch(input_batch, target_batch, model, device): 
    input_batch = input_batch.to(device) 
    target_batch = target_batch.to(device) 
    logits = model(input_batch) 
    loss = torch.nn.functional.cross_entropy( 
        logits.flatten(0, 1), target_batch.flatten() 
    ) 
    return loss 

def calc_loss_loader(data_loader, model, device, num_batches=None): 
    total_loss = 0. 
    if len(data_loader) == 0: 
        return float("nan") 
    elif num_batches is None: 
        num_batches = len(data_loader) 
    else: 
        num_batches = min(num_batches, len(data_loader)) 
    for i, (input_batch, target_batch) in enumerate(data_loader): 
        if i < num_batches: 
            loss = calc_loss_batch( 
                input_batch, target_batch, model, device 
            ) 
            total_loss += loss.item() 
        else: 
            break 
    return total_loss / num_batches

def train_dataTxt():
    # folder_path ="/kaggle/input/datasets/mangzhang/shardstxt3/"
    # folder_path = "/kaggle/input/datasets/mangzhang/qwennewtrainclean/"
    # train_dataset = (
    #     wds.WebDataset(tar_path, shardshuffle=1000) # pyright: ignore[reportAttributeAccessIssue]
    #     .map(decode_txt)      # 👈 用 .map() 来执行自定义的解码逻辑
    #     .to_tuple("text") 
    # )
    # with open("/kaggle/input/datasets/mangzhang/qwennewtrainclean/pretrain_body_clean.txt","r",encoding="utf-8") as f:
        # text_data = f.read()
    # max_length =1024
    # stride = 128
    # dataset = GPTDatasetV1(text_data, tokenizer, max_length, stride)

    train_dataloader = DataLoader(
       train_dataset, 
       batch_size=4, 
       shuffle=True, 
       collate_fn=collate_fn,
       # max_length=GPT_CONFIG_124M["context_length"],
       # stride=GPT_CONFIG_124M["context_length"]
    
    )


    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=0.1 )
    scheduler = CosineAnnealingLR(optimizer, T_max=1000, eta_min=1e-6)
    model.train()
    stop_train = False
    for epoch in range(2):
        # if stop_train == True:
        #     # save_model()
        #     break
        for step, batch in enumerate(train_dataloader):
            optimizer.zero_grad()
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            # loss = calc_loss_batch( 
            #     step, batch, model, device 
            # ) 
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs["loss"]
            loss.backward()
            optimizer.step()
            scheduler.step()
            # plot_chart(loss.item(), epoch)
            if step % 10 == 0:
                print("Time now is ---- ",datetime.now())
                print(f"Epoch {epoch}, Train Loss: {loss.item():.4f}")
                #Grad——Norm：{model.out_head.weight.grad.norm().item()}")
                # if loss.item() < 0.4 :
                #     stop_train = True
                #     save_model()
                #     break
                print(f"========== 第 Step:{step} batch: {batch} 轮训练完成！ ==========")
    print(f"========== 第 epoch: {epoch} 轮训练彻底完成！ ==========")

In [69]:
# def train_data():
#     # folder_path ="/kaggle/input/datasets/mangzhang/shardstxt3/"
#     folder_path = "/kaggle/input/datasets/mangzhang/allchineseliteral-clean/"
#     # train_dataset = (
#     #     wds.WebDataset(tar_path, shardshuffle=1000) # pyright: ignore[reportAttributeAccessIssue]
#     #     .map(decode_txt)      # 👈 用 .map() 来执行自定义的解码逻辑
#     #     .to_tuple("text") 
#     # )
#     train_dataset = KaggleTextDataset(folder_path)

#     train_dataloader = DataLoader(
#         train_dataset, 
#         batch_size=2, 
#         num_workers=0,
#         collate_fn = collate_fn
#     )

#     optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
#     model.train()
#     stop_train = False
#     for epoch in range(1):
#         if stop_train == True:
#             # save_model()
#             break
#         for step, batch in enumerate(train_dataloader):
#             input_ids = batch["input_ids"].to(device)
#             attention_mask = batch["attention_mask"].to(device)
#             labels = batch["labels"].to(device)
#             optimizer.zero_grad()
#             outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
#             loss = outputs["loss"]
#             loss.backward()
#             optimizer.step()
#             if step % 10 == 0:
#                 print("Time now is ---- ",datetime.now())
#                 print(f"Epoch {epoch}, Train Loss: {loss.item():.4f}")
#                 plot_chart(loss.item(), epoch)
#                 #Grad——Norm：{model.out_head.weight.grad.norm().item()}")
#                 if loss.item() < 0.4 :
#                     stop_train = True
#                     save_model()
#                     break

In [70]:
############################Log status############################
# 1. 创建 Logger
logger = logging.getLogger("mainFineTune")
logger.setLevel(logging.INFO)

# 2. 终端 Handler（Rich 负责好看，带时间戳）
console_handler = RichHandler(show_time=True, rich_tracebacks=True)
# 注意：RichHandler 的时间格式也由 datefmt 控制
console_handler.setFormatter(logging.Formatter("%(message)s", datefmt="%Y-%m-%d %H:%M:%S"))

# 3. 文件 Handler（标准格式，带时间戳）
file_handler = logging.FileHandler("app.log", mode="a", encoding="utf-8")
file_formatter = logging.Formatter(
    # 👇 asctime 会自动读取 datefmt 的格式
    "%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
file_handler.setFormatter(file_formatter)

# 4. 挂载
logger.addHandler(console_handler)
logger.addHandler(file_handler)


########################Tensorboard chart########################


# 默认会创建一个 ./runs/ 文件夹来存日志
#writer = SummaryWriter("runs/transformer_finetune_experiment")

#for epoch in range(num_epochs):
    # ... 你的训练逻辑 ...
    
    # 记录训练指标（Scalar 就是标量，比如 loss, acc）
    # writer.add_scalar("Loss/train", train_loss, epoch)
    # writer.add_scalar("Accuracy/train", train_acc, epoch)
    
    # 验证集指标同理
    # writer.add_scalar("Loss/val", val_loss, epoch)
    # writer.add_scalar("Accuracy/val", val_acc, epoch)

# 训练结束后，关闭 writer（或者手动 flush 确保数据写入磁盘）
#writer.close()

In [71]:
#########################Simple Plot chart#########################
plotlosses = PlotLosses()

def plot_chart(train_loss, epoch):
    """
    绘制损失和精度曲线。
    :param loss: 训练损失列表
    :param val_loss: 验证损失列表
    :param train_acc: 训练精度列表
    :param val_acc: 验证精度列表
    :param epoch: 当前训练轮次
    """
    # 绘制损失曲线
    plotlosses.update({
        'train_loss': train_loss, 
        # 'val_loss': val_loss, 
        # 'train_acc': train_acc, 
        # 'val_acc': val_acc,
        'epoch': epoch
        })
    # 绘制精度曲线
    plotlosses.send()
plotlosses = PlotLosses()

#########################Tqdm Progress Bar#########################
# from tqdm import tqdm
# for i, entry in tqdm(enumerate(test_data), total=len(test_data), desc="Processing"):
#     # 处理每个条目
#     input_text = format_input(entry)
#     token_ids = generate(
#         model=model,
#         idx=text_to_token_ids(input_text, tokenizer).to(device),
#         max_new_tokens=256,
#         context_size=BASE_CONFIG["context_length"],
#         eos_id=50256
#     )
#     generated_text = token_ids_to_text(token_ids, tokenizer)
#     response_text = generated_text[len(input_text):].replace("### Response:", "").strip()
#     # 去掉输入部分，只保留生成的回复)
#     test_data[i]["model_response"] = response_text
# with open("instruction-data-with-response.json", "w", encoding="utf-8") as f:
#     json.dump(test_data, f, ensure_ascii=False, indent=4)


############################计算损失批次############################

def calc_loss_batch(input_batch, target_batch, model, device):
    """
    计算一个批次的损失。
    :param input_batch: 输入数据的批次
    :param target_batch: 目标数据的批次
    :param model: 模型
    :param device: 设备（CPU 或 GPU）
    :return: 损失值
    """
    # 将输入和目标数据移动到指定设备
    pass
    # 前向传播
    
    # 获取损失值

    # 返回损失值

In [72]:
############################计算损失加载器############################

def calc_loss_loader(data_loader, model, device, num_batches=None):
    """
    计算整个数据加载器的平均损失。
    :param data_loader: 数据加载器
    :param model: 模型
    :param device: 设备（CPU 或 GPU）
    :param num_batches: 要计算的批次数量（可选）
    :return: 平均损失值
    """
    # 初始化总损失和批次计数
    pass
    # 遍历数据加载器

        # 检查是否需要停止遍历
    
        # 计算当前批次的损失
        
        # 累加损失值
        
        # 增加批次计数

    # 计算平均损失值
    
    # 返回平均损失值

In [73]:
############################训练分类检验############################

def train_classifier_simple(model, train_loader, val_loader, optimizer, device, num_epochs, eval_freq, eval_iter):
    # 初始化训练损失， 验证损失， 训练精度， 验证精度列表 变量
    pass
    # 初始化example_seen， global_step
    
    # 遍历每个epoch
        # 设置模型为训练模式
        
        # 遍历输入，输出训练数据加载器中的每个批次
        
            # 优化器清除梯度

            # 计算损失
            
            # 反向传播
            
            # 优化器更新参数
            
            # track examples instead of tokens
            
            # 步长增加1
            
            
            # 如果步长是eval_freq的倍数
                # 计算训练损失和验证损失
                
                # 增加到训练损失变量
                
                # 增加到验证损失变量
                
                # 打印当前epoch，global_step，训练损失，验证损失
            
            # 每个批次后的精度计算
            # 训练精度计算
            
            # 验证精度计算
            
            # 打印当前epoch的训练精度，验证精度
            
            # 训练精度变量增加此次的训练精度
            
            # 验证精度变量增加此次的验证精度
            
        # 返回训练损失，验证损失，训练精度，验证精度， example_seen
        
        # 在代码里，global_step 是每次循环 +1（代表走了多少步），而 examples_seen 是每次加上 batch_size（代表看了多少数据）。
        # 如果你发现 loss 降得很慢，你可以看 examples_seen。如果 # examples_seen 已经很大了（比如看了十几万条），但 Loss 还不降，那就说明是模型欠拟合或者学习率有问题，而不是训练时间不够。

In [74]:
############################评估模式############################

def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    """
    评估模型在训练集和验证集上的性能。
    :param model: 模型
    :param train_loader: 训练数据加载器
    :param val_loader: 验证数据加载器
    :param device: 设备（CPU 或 GPU）
    :param eval_iter: 评估的批次数量
    :return: 训练集和验证集的平均损失值
    """
    # 设置模型为评估模式
    pass
    # 禁止梯度下降，计算训练集的平均损失
    
    # 计算验证集的平均损失
    
    # 训练模式
    
    # 返回训练集和验证集的平均损失值



############################优化器设计############################
# 学习率设计

# 优化器设计
optimizer  = {}

# 训练轮次初始化

# 训练分类器的简单训练函数

# 打印训练损失，验证损失，训练精度，验证精度， example_seen， global_step， examples_seen

############################SAVE MODEL############################
import re
def save_model():
    
 # 1. 获取当前时间戳（比如 20261113_153022）
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # 2. 把你配置里的一些关键参数也带在文件名里（可选）
    model_size = NEW_CONFIG.get("n_layers", 12)
    emb_dim = NEW_CONFIG.get("emb_dim", 768)
    
    # 3. 组装文件名：包含时间戳，绝对不会覆盖
    # file_name = f"allChineseLiteral_L{model_size}_E{emb_dim}_{timestamp}.pth"
    file_name = f"allChineseLiteralCN.pth"
    # 或者如果你想把原来的 CHOOSE_MODEL 恢复：
    # file_name = f"{re.sub(r'[ ()]', '', model_name)}_sft.pth"
    if os.path.exists(file_name):
        os.remove(file_name)
        print(f"旧的 {file_name} 已删除。")
    else:
        print(f"{file_name} 不存在，无需删除。")
    torch.save(model.state_dict(), file_name)
    logger.info(f"Model saved to {file_name}")




##########################MAIN############################

# 按装订区域中的绿色按钮以运行脚本。
if __name__ == '__main__':
    status = "Training"
    # logger.info("Starting the prepare data...")
    # prepare_data()
    logger.info("Starting the train data...")
    console.print("Training data...", style='blue')
    # train_dataTxt()
    logger.info("Starting the save model...")
    # save_model()
    
    if status == "Training":
        # prepare_data()
        pass
    else:   
        #  check_if_running("ollama")
        pass

    # generate_model_scores(test_data, "model_response")

2026-11-18 23:42:52 INFO     Starting the train data...                                            ]8;id=641327;file:///tmp/ipykernel_50/1444669983.py\1444669983.py]8;;\:]8;id=455208;file:///tmp/ipykernel_50/1444669983.py#71\71]8;;\

Training data...

                    INFO     Starting the save model...                                            ]8;id=300889;file:///tmp/ipykernel_50/1444669983.py\1444669983.py]8;;\:]8;id=238130;file:///tmp/ipykernel_50/1444669983.py#74\74]8;;\

In [75]:
model=GPT_MODEL(NEW_CONFIG)
# model.to(device)
# model.load_state_dict(torch.load("allChineseLiteralCN.pth", map_location=torch.device('cpu')))
model.eval()

GPT_MODEL(
  (tok_emb): Embedding(50260, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): ModuleList(
    (0-11): 12 x TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
  )
  (final_norm): LayerNorm()
  (out_head): Linear(in_features=768, ou

In [76]:
text = "话说天下大势,分久必合,合久必分。"
ids = tokenizer.encode(text)
print(len(ids), ids[:20])
print(repr(tokenizer.decode(ids)))     # 必须原样还原!
ids = tokenizer.encode("话说天下大势,分久必合,合久必分。")
print(repr([tokenizer.decode([i]) for i in ids]))


29 [46237, 251, 46237, 112, 25465, 10310, 233, 32014, 27950, 123, 11, 26344, 228, 20046, 227, 33232, 227, 28938, 230, 11]
'话说天下大势,分久必合,合久必分。'
['�', '�', '�', '�', '天', '�', '�', '大', '�', '�', ',', '�', '�', '�', '�', '�', '�', '�', '�', ',', '�', '�', '�', '�', '�', '�', '�', '�', '。']


In [77]:
def generate_text_simple9(model, idx,max_new_tokens, context_size): 
    for _ in range(max_new_tokens): 
        idx_cond = idx[:, -context_size:] 
        with torch.no_grad(): 
            logits = model(idx_cond) 
        logits = logits[:, -1, :] 
        probas = torch.softmax(logits, dim=-1) 
        idx_next = torch.argmax(probas, dim=-1, keepdim=True) 
        idx = torch.cat((idx, idx_next), dim=1) 
    return idx 

In [78]:
def text_to_token_ids(text, tokenizer): 
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'}) 
    encoded_tensor = torch.tensor(encoded).unsqueeze(0) 
    return encoded_tensor 
def token_ids_to_text(token_ids, tokenizer): 
    flat = token_ids.squeeze(0) 
    return tokenizer.decode(flat.tolist()) 
token_ids = generate_text_simple9( 
    model=model, 
    idx=text_to_token_ids("孙悟空", tokenizer), 
    max_new_tokens=25, 
    context_size=GPT_CONFIG_124M["context_length"] 
) 
print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

KeyError: (slice(None, None, None), -1, slice(None, None, None))

In [ ]:
def generate_text(prompt, max_new_tokens=100):
    # model.eval()
    input_ids = tokenizer.encode(prompt, allowed_special={"<|endoftext|>", "<|persona|>", "<|user|>", "<|bot|>"}
)
    input_tensor = torch.tensor([input_ids]).to(device)
    top_k = 50

    for _ in range(max_new_tokens):
        with torch.no_grad():
            outputs = model(input_ids=input_tensor)
            logits = outputs["logits"]
            next_token_logits = logits[:, -1, :] / 0.8
            top_logits, top_indices = torch.topk(next_token_logits, top_k)
            
            next_token_id = torch.argmax(next_token_logits, dim=-1).unsqueeze(0)
            input_tensor = torch.cat([input_tensor, next_token_id], dim=-1)
    return tokenizer.decode(input_tensor[0].tolist())

In [ ]:
def generate_text2(prompt, max_new_tokens=100):
    model.eval()
    allowed_special = {"<|endoftext|>","<|persona|>","<|user|>","<|bot|>"}
    input_ids = tokenizer.encode(
        prompt,
        allowed_special=allowed_special
    )
    input_tensor = torch.tensor([input_ids]).to(device)
    top_k = 50
    eos_id = tokenizer.encode("<|endoftext|>", allowed_special=allowed_special)[0]

    for _ in range(max_new_tokens):
        with torch.no_grad():
            idx_cond = input_tensor[:, -GPT_CONFIG_124M["context_length"]:]
            outputs = model(input_ids=idx_cond)
            logits = outputs["logits"]
            next_token_logits = logits[:, -1, :] / 0.8

            top_logits, top_indices = torch.topk(next_token_logits, top_k)
            mask = torch.full_like(next_token_logits, -float("inf"))
            mask.scatter_(1, top_indices, next_token_logits.gather(1, top_indices))
            next_token_logits = mask

            probas = torch.softmax(next_token_logits, dim=-1)
            next_token_id = torch.multinomial(probas, num_samples=1)

        input_tensor = torch.cat([input_tensor, next_token_id], dim=-1)
        if next_token_id.item() == eos_id:
            break

    return tokenizer.decode(input_tensor[0].tolist())


In [ ]:
system_msg = "<|persona|>你是孙悟空。"
user_msg = "<|user|>话说唐僧<|user|>"
prompt2 = f"{system_msg}{user_msg}<|bot|>"

full_text = generate_text2(prompt2)
# 提取bot回答，去掉前面prompt、后面eos
ans = full_text.split("<|bot|>")[-1].split("<|endoftext|>")[0]
print("完整输出：", full_text)
print("纯净回答：", ans)


In [ ]:
# def generate_text_simple(model, idx, 
#                          max_new_tokens, context_size): 
#     for _ in range(max_new_tokens): 
#         idx_cond = idx[:, -context_size:] 
#         with torch.no_grad(): 
#             logits = model(idx_cond) 
#         logits = logits[:, -1, :] 
#         probas = torch.softmax(logits, dim=-1) 
#         idx_next = torch.argmax(probas, dim=-1, keepdim=True) 
#         idx = torch.cat((idx, idx_next), dim=1) 
#     return idx

# start_context = "Hello, I am" 
# encoded = tokenizer.encode(start_context) 
# print("encoded:", encoded) 
# encoded_tensor = torch.tensor(encoded).unsqueeze(0) 
# print("encoded_tensor.shape:", encoded_tensor.shape) 


# out = generate_text_simple( 
#     model=model, 
#     idx=encoded_tensor, 
#     max_new_tokens=6, 
#     context_size=GPT_CONFIG_124M["context_length"] 
# ) 
# print("Output:", out) 
# print("Output length:", len(out[0]))

In [ ]:
# import torch

# def generate_safe(prompt, max_new_tokens=30, temperature=0.1, top_k=50):
#     model.eval()
#     input_ids = tokenizer.encode(prompt)
#     input_tensor = torch.tensor([input_ids]).to(device)
    
#     # 限制输入长度，防止越界
#     if input_tensor.shape[1] > 512:
#         input_tensor = input_tensor[:, -512:]
        
#     for _ in range(max_new_tokens):
#         with torch.no_grad():
#             outputs = model(input_ids=input_tensor)
#             logits = outputs["logits"]
            
#             # 1. 取最后一个词，并除以温度
#             next_token_logits = logits[:, -1, :] / temperature
            
#             # 2. Top-K 过滤
#             if top_k is not None:
#                 top_logits, _ = torch.topk(next_token_logits, top_k)
#                 next_token_logits[next_token_logits < top_logits[:, -1]] = float('-inf')
            
#             # 3. Softmax 转概率
#             probas = torch.softmax(next_token_logits, dim=-1)
            
#             # 4. 随机采样（打破死循环）
#             next_token_id = torch.multinomial(probas, num_samples=1)
            
#             input_tensor = torch.cat([input_tensor, next_token_id], dim=-1)
            
#     # 过滤掉特殊 token 再解码

#     output_ids = input_tensor[0].tolist()
#     print("Output ids: ", output_ids)
#     output_text = tokenizer.decode(output_ids)
#     return output_text

# # 测试
# print("测试1:", generate_safe("如来佛祖"))
# print("测试2:", generate_safe("金角大王"))

In [ ]:
# # 在 train_lora 函数开头测试
# checkpoint = torch.load("allChineseLiteralCN.pth", map_location="cpu")
# print("权重文件的键名:", list(checkpoint.keys())[:10])
# # 在 train_lora 函数开头测试
# checkpoint = torch.load("lora_sft_weights.pth", map_location="cpu")
# print("权重文件的键名:", list(checkpoint.keys())[:10])

In [ ]:
# 确保你的环境里已经定义了 GPT_MODEL、tiktoken 的 tokenizer
# train_lora(
#     json_path="/kaggle/input/datasets/mangzhang/sf-data/sft_data.json", 
#     base_weight_path="allChineseLiteralCN.pth", 
#     tokenizer=tokenizer
# )

In [ ]:
##Train sft lora OK

import torch
import torch.nn as nn
import math
import json
import os
from torch.utils.data import Dataset, DataLoader




# ===================== 2. LoRA 核心类 =====================
class LoRALayer(nn.Module):
    def __init__(self, in_dim, out_dim, rank=8, alpha=16):
        super().__init__()
        self.scaling = alpha / rank
        # A 矩阵用正态分布初始化，必须非常小
        self.A = nn.Parameter(torch.randn(in_dim, rank) * 0.001)
        # B 矩阵初始化为 0
        self.B = nn.Parameter(torch.zeros(rank, out_dim))

    def forward(self, x):
        return (x @ self.A @ self.B) * self.scaling

class LinearWithLoRA(nn.Module):
    def __init__(self, linear, rank=8, alpha=16):
        super().__init__()
        self.linear = linear
        # 冻结原始权重
        for param in self.linear.parameters():
            param.requires_grad = False
        self.lora = LoRALayer(linear.in_features, linear.out_features, rank, alpha)

    def forward(self, x):
        return self.linear(x) + self.lora(x)

# 递归替换模型中所有的 Linear 层
def apply_lora_to_model(model, rank=8, alpha=16):
    for name, module in model.named_children():
        if isinstance(module, nn.Linear):
            setattr(model, name, LinearWithLoRA(module, rank, alpha))
        elif isinstance(module, (nn.ModuleList, nn.Sequential)):
            apply_lora_to_model(module, rank, alpha)
        else:
            apply_lora_to_model(module, rank, alpha)
    return model

# ===================== 3. Dataset 与 Collate =====================
class SFTDataset(Dataset):
    def __init__(self, json_path, tokenizer):
        with open(json_path, "r", encoding="utf-8") as f:
            self.data = json.load(f)
        self.tokenizer = tokenizer
        self.allowed_special = {"<|persona|>", "<|user|>", "<|bot|>", "<|endoftext|>"}

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        sys_msg = item.get("system", "你是一个历史专家")
        user_msg = item.get("instruction", "")
        bot_msg = item.get("output", "")
        
        text = f"<|persona|>{sys_msg}<|user|>{user_msg}<|bot|>{bot_msg}<|endoftext|>"
        ids = self.tokenizer.encode(text, allowed_special=self.allowed_special)
        return torch.tensor(ids, dtype=torch.long)

def collate_fn(batch):
    max_len = max(len(ids) for ids in batch)
    max_len = min(max_len, 512)  # 微调序列缩短到 512 防显存爆炸
    pad_id = 50256
    bot_token_id = 50259
    
    padded_ids, masks, labels = [], [], []
    
    for ids in batch:
        ids = ids[:max_len]
        seq_len = len(ids)
        pad_len = max_len - seq_len
        
        padded = ids.tolist() + [pad_id] * pad_len
        mask = [1] * seq_len + [0] * pad_len
        
        label = [-100] * max_len
        bot_idx = (ids == bot_token_id).nonzero()
        if len(bot_idx) > 0:
            start_idx = bot_idx[0].item()
            for i in range(start_idx, seq_len):
                label[i] = ids[i].item()
                
        padded_ids.append(padded)
        masks.append(mask)
        labels.append(label)
        # labels = torch.cat([labels[:, 1:],
        #             torch.full((labels.size(0), 1), -100, dtype=torch.long)], dim=1)
        
    return {
        "input_ids": torch.tensor(padded_ids, dtype=torch.long),
        "attention_mask": torch.tensor(masks, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long)
    }

# ===================== 4. 训练主函数 =====================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_lora(json_path, base_weight_path, tokenizer):
    # 1. 加载基础模型（你的 GPT_MODEL）
    model = GPT_MODEL(NEW_CONFIG)
    # 加载预训练的 50260 词表权重
    model.load_state_dict(torch.load(base_weight_path, map_location="cpu"))
    print("✅ 基础模型加载完成")

    # 2. 全量挂载 LoRA（把每一层的 Linear 都套上外挂）
    apply_lora_to_model(model, rank=8)
    model.to(device)
    model.train()

    # 3. 检查可训练参数
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"✅ LoRA 挂载完成，可训练参数量: {trainable_params:,}")

    # 4. 准备数据和优化器
    dataset = SFTDataset(json_path, tokenizer)
    dataloader = DataLoader(dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)
    
    # 极其重要：学习率必须降到 1e-5
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-5)
    scheduler = CosineAnnealingLR(optimizer, T_max=1000, eta_min=1e-6)
    num_epochs = 1
    for epoch in range(num_epochs):
        total_loss = 0
        for step, batch in enumerate(dataloader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
          
            # 防呆检查：如果全是 -100，跳过
            # if (labels != -100).sum().item() == 0:
            #     continue

            optimizer.zero_grad()
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs["loss"]

            # 强行拦截 NaN
            if torch.isnan(loss) or torch.isinf(loss):
                print(f"⚠️ Step {step} Loss 变 NaN，已拦截！")
                # optimizer.zero_grad()
                continue

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()
            if step % 100 == 0:
                print(f"Epoch {epoch+1}, Step {step}, Loss: {loss.item():.4f}")
                
        print(f"========== Epoch {epoch+1} 完成，平均 Loss: {total_loss / len(dataloader):.4f} ==========")

    # 6. 保存 LoRA 权重
    lora_state = {k: v for k, v in model.state_dict().items() if "lora" in k}
    torch.save(lora_state, "lora_sft_weights.pth")
    print("🎉 LoRA 微调完成，已保存 lora_sft_weights.pth")

In [ ]:
json_path = "/kaggle/input/datasets/mangzhang/cnliteralconversation/allChineseLiteral_sft_custom.json"
# base_weight_path = "allChineseLiteralCN.pth"
base_weight_path = "/kaggle/input/datasets/mangzhang/allchineseliteralcn/allChineseLiteralCN.pth"
# train_lora(json_path,base_weight_path,tokenizer)

In [79]:

import torch

def load_model_with_lora(base_model_path, lora_path, config):
    # 1. 先初始化原版模型骨架
    model = GPT_MODEL(NEW_CONFIG)
    
    # 2. 加载基础大模型权重（西游记预训练权重）
    model.load_state_dict(torch.load(base_model_path, map_location="cpu"))
    print("✅ 基础模型加载完成")
    
    # 3. 重新把模型里的层替换成带 LoRA 的层（必须和训练时结构一样）
    model = apply_lora_to_model(model, rank=8)
    
    # 4. 把微调好的 LoRA 权重“挂载”上去
    lora_weights = torch.load(lora_path, map_location="cpu")
    model.load_state_dict(lora_weights, strict=False) # strict=False 允许忽略基础权重
    print("✅ LoRA 微调权重挂载完成")
    
    # 5. 移到设备并开启测试模式
    model.to(device)
    model.eval()
    return model
# # 使用：
# config = GPT_CONFIG_124M
model = load_model_with_lora("/kaggle/input/datasets/mangzhang/allchineseliteralcn/allChineseLiteralCN.pth", "lora_sft_weights.pth", NEW_CONFIG)

#######################################################################################
def generate_text(model, prompt, max_new_tokens=50):
    # 编码时记得带上特殊 Token
    input_ids = tokenizer.encode(prompt, allowed_special={"<|persona|>", "<|user|>", "<|bot|>"})
    input_tensor = torch.tensor([input_ids]).to(device)
    
    for _ in range(max_new_tokens):
        with torch.no_grad():
            idx_cond = input_tensor[:, -1024:]
            outputs = model(input_ids=idx_cond)
            logits = outputs["logits"][:, -1, :] / 0.7
            
            # Top-K 过滤
            top_logits, _ = torch.topk(logits, 50)
            logits[logits < top_logits[:, -1]] = float('-inf')
            
            probas = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(probas, num_samples=1)
            
            input_tensor = torch.cat([input_tensor, next_token], dim=-1)
            
            if next_token.item() == 50259: # 遇到 <|endoftext|> 停止
                break
                
    return tokenizer.decode(input_tensor[0].tolist())

# 测试

✅ 基础模型加载完成
✅ LoRA 微调权重挂载完成


In [81]:
prompt = "刘备是谁？"
print(generate_text(model, prompt))

刘备是谁？<|bot|>


In [93]:
import json
import torch

#######################################################################################
# 构造 messages
#######################################################################################
# 直接定义 s，一条样本
s = {
    "system": "你是熟读三国演义的助理，回答简洁准确",
    "instruction": "刘备是谁？",
    "output": "刘备，字玄德，蜀汉开国皇帝。"
}

if isinstance(s, str):
    s = json.loads(s)
if isinstance(s, list):
    s = s[0]

messages = [
    {"role": "system", "content": s["system"]},
    {"role": "user",   "content": s["instruction"]},
    {"role": "assistant", "content": s["output"]},
]

#######################################################################################

def load_model_with_lora(base_model_path, lora_path, config):
    # 1. 先初始化原版模型骨架
    model = GPT_MODEL(NEW_CONFIG)

    # 2. 加载基础大模型权重（西游记预训练权重）
    model.load_state_dict(torch.load(base_model_path, map_location="cpu"))
    print("✅ 基础模型加载完成")

    # 3. 重新把模型里的层替换成带 LoRA 的层（必须和训练时结构一样）
    model = apply_lora_to_model(model, rank=8)

    # 4. 把微调好的 LoRA 权重"挂载"上去
    lora_weights = torch.load(lora_path, map_location="cpu")
    model.load_state_dict(lora_weights, strict=False)
    print("✅ LoRA 微调权重挂载完成")

    # 5. 移到设备并开启测试模式
    model.to(device)
    # model.eval()   # ← 你原来有这行
    return model

model = load_model_with_lora("/kaggle/input/datasets/mangzhang/allchineseliteralcn/allChineseLiteralCN.pth", "lora_sft_weights.pth", NEW_CONFIG)

#######################################################################################

def generate_text(model, prompt, max_new_tokens=500):
    model.eval()  # ✅ 关键修复：确保 dropout 关闭（防止 training=True 导致乱码）

    # 编码时记得带上特殊 Token
    input_ids = tokenizer.encode(prompt, allowed_special={"<|persona|>", "<|user|>", "<|bot|>"})
    input_tensor = torch.tensor([input_ids]).to(device)

    for _ in range(max_new_tokens):
        with torch.no_grad():
            idx_cond = input_tensor[:, -1024:]
            outputs = model(input_ids=idx_cond)
            logits = outputs["logits"][:, -1, :] / 0.3

            # Top-K 过滤
            top_logits, _ = torch.topk(logits, 20)
            logits[logits < top_logits[:, -1]] = float('-inf')

            probas = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(probas, num_samples=1)

            input_tensor = torch.cat([input_tensor, next_token], dim=-1)

            if next_token.item() == 50256:
                break

    return tokenizer.decode(input_tensor[0].tolist())

# 测试
# prompt = "<|persona|>" + s["system"] + "<|user|>" + s["instruction"] + "<|bot|>"
prompt = "<|persona|>你熟读红楼梦,西游记,水浒传,三国演义,聊斋志异,喻世明言,警世通言,醒世恒言,初刻拍案惊奇，二刻拍案惊奇，回答尽量贴合原著，语言偏古风简洁，依据书中内容作答。<|user|>请简述孙悟空大闹天宫的经过？<|bot|>"
result = generate_text(model, prompt, max_new_tokens=500)
print(result)

✅ 基础模型加载完成
✅ LoRA 微调权重挂载完成
<|persona|>你是孙悟空。<|user|>孙悟空大闹天宫？<|bot|>曰，曰，曰，�，可�，马见，曰，曰，见“至，赶，至，曰，�“见”�，后，�“见见“汱��，母，又�，起，曰，小，见，见“至，自曰，�“，见”�曰，见“，曰，曰，又，见”见“，曰，曰，见“�”其，只见“，自见�”自�”起，曰，曰，见“曰，曰，�”�，九曰，把��。“见”曰，自见“曰，曰，自，曰，曰，见�”走，见“曰，曰，曰，曰，曰，马，把，计，�，不曰，自曰，曰，曰，曰，曰，�


In [ ]:
# prompt = "<|persona|>你是<|user|>刘备是哪里人<|bot|>"
prompt="<|persona|>你是三国刘备<|user|>简述时有宦官曹节等弄权，窦武、陈蕃谋诛之，机事不这段故事的经过？<|bot|>"
# prompt="隔了两日，只见王三又来索取那刘家四百两银子利这个情节发生的背景是什么？"

print(generate_text(model, prompt, max_new_tokens=500))




In [ ]:
ids = tokenizer.encode("话说天下大势,分久必合,合久必分。")
print([tokenizer.decode([i]) for i in ids])


In [ ]:
# ###Qwen
# import os
# import json
# import torch

# #######################################################################################
# # Kaggle 路径工具
# #######################################################################################

# def find_file(filename):
#     """
#     在 Kaggle 常见目录里找文件:
#     1. 当前目录
#     2. /kaggle/working
#     3. /kaggle/input 下递归查找
#     """
#     candidates = [
#         filename,
#         os.path.join("/kaggle/working", filename),
#     ]

#     for p in candidates:
#         if os.path.exists(p):
#             return p

#     for root, dirs, files in os.walk("/kaggle/input"):
#         if filename in files:
#             return os.path.join(root, filename)

#     raise FileNotFoundError(f"找不到文件: {filename}")


# BASE_MODEL_PATH = find_file("allChineseLiteralCN.pth")
# LORA_PATH = find_file("lora_sft_weights.pth")

# print("基础模型路径:", BASE_MODEL_PATH)
# print("LoRA 权重路径:", LORA_PATH)

# #######################################################################################
# # 设备
# #######################################################################################

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print("使用设备:", device)

# #######################################################################################
# # 构造测试样本 s
# #######################################################################################

# if isinstance(s, str):
#     s = json.loads(s)

# if isinstance(s, list):
#     s = s[0]

# messages = [
#     {"role": "system", "content": s["system"]},
#     {"role": "user", "content": s["instruction"]},
#     {"role": "assistant", "content": s["output"]},
# ]

# #######################################################################################
# # state_dict 清理工具
# #######################################################################################

# def unwrap_state_dict(obj):
#     """
#     兼容这些保存格式:
#     1. 直接保存 model.state_dict()
#     2. {"model_state_dict": state_dict}
#     3. {"state_dict": state_dict}
#     4. {"model": state_dict}
#     """
#     if isinstance(obj, dict):
#         for key in ["model_state_dict", "state_dict", "model"]:
#             if key in obj and isinstance(obj[key], dict):
#                 return obj[key]
#     return obj


# def clean_state_dict_keys(sd):
#     """
#     去掉 DDP / torch.compile 可能带来的前缀。
#     不改变模型结构，只清理 key 名。
#     """
#     new_sd = {}
#     for k, v in sd.items():
#         k = k.replace("module.", "")
#         k = k.replace("_orig_mod.", "")
#         new_sd[k] = v
#     return new_sd


# #######################################################################################
# # 特殊 token 工具
# #######################################################################################

# SPECIAL_TOKENS = {
#     "<|persona|>",
#     "<|user|>",
#     "<|bot|>",
#     "<|endoftext|>",
# }


# def encode_text(text):
#     return tokenizer.encode(text, allowed_special=SPECIAL_TOKENS)


# def get_single_token_id(token):
#     """
#     如果 token 是单个特殊 token，返回它的 id。
#     如果不是单 token，返回 None。
#     """
#     try:
#         ids = tokenizer.encode(token, allowed_special={token})
#         if len(ids) == 1:
#             return ids[0]
#     except Exception:
#         pass
#     return None


# EOS_ID = get_single_token_id("<|endoftext|>")
# BOT_ID = get_single_token_id("<|bot|>")
# USER_ID = get_single_token_id("<|user|>")
# PERSONA_ID = get_single_token_id("<|persona|>")

# print("<|endoftext|> id:", EOS_ID)
# print("<|bot|> id:", BOT_ID)
# print("<|user|> id:", USER_ID)
# print("<|persona|> id:", PERSONA_ID)

# # 遇到这些 token 就停止生成
# # 注意：你之前写死 50259，很可能其实是 <|bot|>，不是 <|endoftext|>
# STOP_IDS = set()

# if EOS_ID is not None:
#     STOP_IDS.add(EOS_ID)

# if BOT_ID is not None:
#     STOP_IDS.add(BOT_ID)

# if USER_ID is not None:
#     STOP_IDS.add(USER_ID)

# if PERSONA_ID is not None:
#     STOP_IDS.add(PERSONA_ID)

# print("停止 token ids:", STOP_IDS)

# #######################################################################################
# # LoRA 加载函数：不修改模型结构，只修复 apply_lora 返回 None 和权重校验
# #######################################################################################

# def load_model_with_lora(base_model_path, lora_path, config):
#     # 1. 初始化原始 GPT 模型骨架
#     model = GPT_MODEL(config)

#     # 2. 加载基础模型权重
#     base_sd = torch.load(base_model_path, map_location="cpu")
#     base_sd = unwrap_state_dict(base_sd)
#     base_sd = clean_state_dict_keys(base_sd)

#     model.load_state_dict(base_sd, strict=True)
#     print("✅ 基础模型加载完成")

#     # 3. 读取 LoRA 权重
#     lora_sd = torch.load(lora_path, map_location="cpu")
#     lora_sd = unwrap_state_dict(lora_sd)
#     lora_sd = clean_state_dict_keys(lora_sd)

#     lora_keys_in_ckpt = [k for k in lora_sd.keys() if "lora" in k.lower()]
#     print("LoRA checkpoint 中 LoRA 参数数量:", len(lora_keys_in_ckpt))
#     print("LoRA checkpoint 前 5 个 key:", list(lora_sd.keys())[:5])

#     if len(lora_keys_in_ckpt) == 0:
#         raise RuntimeError("❌ LoRA 文件中没有发现 lora 参数，请检查 lora_sft_weights.pth")

#     # 4. 调用你原来的 apply_lora_to_model
#     # 关键修复：
#     # 如果你的函数是原地修改并返回 None，则继续使用原 model
#     # 如果你的函数返回新 model，则使用返回值
#     old_model = model
#     ret = apply_lora_to_model(model, rank=8)

#     if ret is None:
#         print("⚠️ apply_lora_to_model 返回 None，按“原地修改模型”处理")
#         model = old_model
#     else:
#         print("✅ apply_lora_to_model 返回了新模型，使用返回值")
#         model = ret

#     # 5. 检查模型里现在有没有 LoRA 参数
#     model_sd = model.state_dict()
#     model_lora_keys = [k for k in model_sd.keys() if "lora" in k.lower()]

#     print("模型中 LoRA 参数数量:", len(model_lora_keys))
#     print("模型中 LoRA 前 5 个 key:", model_lora_keys[:5])

#     if len(model_lora_keys) == 0:
#         raise RuntimeError(
#             "❌ apply_lora_to_model 没有把 LoRA 挂到模型上。\n"
#             "这通常说明 apply_lora_to_model 内部没有正确替换层，或者函数没有 return model。\n"
#             "在不改模型结构的前提下，无法继续正确加载 LoRA。\n"
#             "请检查 apply_lora_to_model 源码。"
#         )

#     # 6. 检查 LoRA key 是否能对上
#     model_keys = set(model_sd.keys())
#     missing_in_model = [k for k in lora_sd.keys() if k not in model_keys]

#     if missing_in_model:
#         print("❌ 以下 LoRA checkpoint key 在模型中找不到，前 20 个:")
#         for k in missing_in_model[:20]:
#             print("  ", k)

#         raise RuntimeError(
#             "❌ LoRA key 不匹配，不能继续假装加载成功。\n"
#             "请保证推理时 apply_lora_to_model 的结构和训练时完全一致。"
#         )

#     # 7. 真正加载 LoRA 权重
#     result = model.load_state_dict(lora_sd, strict=False)

#     missing = result.missing_keys
#     unexpected = result.unexpected_keys

#     missing_lora = [k for k in missing if "lora" in k.lower()]
#     unexpected_lora = [k for k in unexpected if "lora" in k.lower()]

#     print("missing keys 总数:", len(missing))
#     print("unexpected keys 总数:", len(unexpected))
#     print("missing LoRA keys 数量:", len(missing_lora))
#     print("unexpected LoRA keys 数量:", len(unexpected_lora))

#     if missing_lora or unexpected_lora:
#         print("missing_lora 前 10 个:", missing_lora[:10])
#         print("unexpected_lora 前 10 个:", unexpected_lora[:10])
#         raise RuntimeError("❌ LoRA 权重没有完整加载")

#     print(f"✅ LoRA 微调权重挂载完成，成功加载 {len(lora_keys_in_ckpt)} 个 LoRA 参数")

#     # 8. 检查 LoRA B 是否非零
#     b_stats = []
#     for n, p in model.named_parameters():
#         if "lora" in n.lower() and n.endswith("B"):
#             b_stats.append((n, p.detach().abs().sum().item()))

#     nonzero_b = sum(1 for _, v in b_stats if v > 1e-8)

#     print(f"LoRA B 非零数量: {nonzero_b}/{len(b_stats)}")

#     if len(b_stats) > 0 and nonzero_b == 0:
#         print("⚠️ 所有 LoRA B 都是 0，说明 LoRA 可能没有训练到，或者加载的仍是初始 LoRA 权重")

#     # 9. 移到设备并 eval
#     model.to(device)
#     model.eval()

#     return model


# model = load_model_with_lora(
#     BASE_MODEL_PATH,
#     LORA_PATH,
#     NEW_CONFIG
# )

# #######################################################################################
# # 生成函数
# #######################################################################################

# @torch.no_grad()
# def generate_text(
#     model,
#     prompt,
#     max_new_tokens=300,
#     temperature=0.7,
#     top_k=50,
#     greedy=False,
#     only_answer=True,
#     context_size=1024,
# ):
#     """
#     only_answer=True:
#         只返回 <|bot|> 后面的新生成内容。
#     only_answer=False:
#         返回 prompt + 生成内容。
#     """

#     model.eval()

#     input_ids = encode_text(prompt)
#     prompt_len = len(input_ids)

#     input_tensor = torch.tensor([input_ids], dtype=torch.long, device=device)

#     generated_ids = []

#     for _ in range(max_new_tokens):
#         idx_cond = input_tensor[:, -context_size:]

#         outputs = model(input_ids=idx_cond)
#         logits = outputs["logits"][:, -1, :]

#         if greedy:
#             next_token = torch.argmax(logits, dim=-1, keepdim=True)
#         else:
#             temperature = max(float(temperature), 1e-6)
#             logits = logits / temperature

#             if top_k is not None and top_k > 0:
#                 k = min(top_k, logits.size(-1))
#                 top_logits, _ = torch.topk(logits, k)
#                 logits[logits < top_logits[:, [-1]]] = float("-inf")

#             probas = torch.softmax(logits, dim=-1)
#             next_token = torch.multinomial(probas, num_samples=1)

#         next_id = next_token.item()

#         if next_id in STOP_IDS:
#             break

#         generated_ids.append(next_id)

#         input_tensor = torch.cat([input_tensor, next_token], dim=-1)

#     if only_answer:
#         return tokenizer.decode(generated_ids)
#     else:
#         return tokenizer.decode(input_tensor[0].tolist())


# #######################################################################################
# # 测试 prompt
# #######################################################################################

# prompt = (
#     "<|persona|>"
#     + s["system"]
#     + "<|user|>"
#     + s["instruction"]
#     + "<|bot|>"
# )

# print("=" * 60)
# print("【输入 prompt】")
# print("=" * 60)
# print(prompt)

# print()
# print("=" * 60)
# print("【测试1】贪心解码")
# print("=" * 60)
# print(generate_text(
#     model,
#     prompt,
#     max_new_tokens=300,
#     greedy=True,
#     only_answer=True
# ))

# print()
# print("=" * 60)
# print("【测试2】temperature=0.7, top_k=50")
# print("=" * 60)
# print(generate_text(
#     model,
#     prompt,
#     max_new_tokens=300,
#     temperature=0.7,
#     top_k=50,
#     only_answer=True
# ))

# print()
# print("=" * 60)
# print("【测试3】temperature=0.3, top_k=30")
# print("=" * 60)
# print(generate_text(
#     model,
#     prompt,
#     max_new_tokens=300,
#     temperature=0.3,
#     top_k=30,
#     only_answer=True
# ))


In [ ]:
# ###Qwen
# import os
# import json
# import torch

# #######################################################################################
# # 路径工具
# #######################################################################################

# def find_file(filename):
#     candidates = [
#         filename,
#         os.path.join("/kaggle/working", filename),
#     ]

#     for p in candidates:
#         if os.path.exists(p):
#             return p

#     if os.path.exists("/kaggle/input"):
#         for root, dirs, files in os.walk("/kaggle/input"):
#             if filename in files:
#                 return os.path.join(root, filename)

#     raise FileNotFoundError(f"找不到文件: {filename}")


# BASE_MODEL_PATH = find_file("allChineseLiteralCN.pth")
# LORA_PATH = find_file("lora_sft_weights.pth")

# print("基础模型路径:", BASE_MODEL_PATH)
# print("LoRA 权重路径:", LORA_PATH)

# #######################################################################################
# # 设备
# #######################################################################################

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print("使用设备:", device)

# #######################################################################################
# # 特殊 token
# #######################################################################################

# SPECIAL_TOKENS = {
#     "<|persona|>",
#     "<|user|>",
#     "<|bot|>",
#     "<|endoftext|>",
# }


# def encode_text(text):
#     return tokenizer.encode(text, allowed_special=SPECIAL_TOKENS)


# def get_single_token_id(token):
#     try:
#         ids = tokenizer.encode(token, allowed_special={token})
#         if len(ids) == 1:
#             return ids[0]
#     except Exception:
#         pass
#     return None


# EOS_ID = get_single_token_id("<|endoftext|>")
# BOT_ID = get_single_token_id("<|bot|>")
# USER_ID = get_single_token_id("<|user|>")
# PERSONA_ID = get_single_token_id("<|persona|>")

# print("<|endoftext|> id:", EOS_ID)
# print("<|bot|> id:", BOT_ID)
# print("<|user|> id:", USER_ID)
# print("<|persona|> id:", PERSONA_ID)

# STOP_IDS = set()

# if EOS_ID is not None:
#     STOP_IDS.add(EOS_ID)

# if BOT_ID is not None:
#     STOP_IDS.add(BOT_ID)

# if USER_ID is not None:
#     STOP_IDS.add(USER_ID)

# if PERSONA_ID is not None:
#     STOP_IDS.add(PERSONA_ID)

# print("停止 token ids:", STOP_IDS)

# #######################################################################################
# # state_dict 工具
# #######################################################################################

# def unwrap_state_dict(obj):
#     if isinstance(obj, dict):
#         for key in ["model_state_dict", "state_dict", "model"]:
#             if key in obj and isinstance(obj[key], dict):
#                 return obj[key]
#     return obj


# def clean_state_dict_keys(sd):
#     new_sd = {}
#     for k, v in sd.items():
#         k = k.replace("module.", "")
#         k = k.replace("_orig_mod.", "")
#         new_sd[k] = v
#     return new_sd


# #######################################################################################
# # 构造 prompt：必须和训练时一致
# #######################################################################################

# SYSTEM_TEXT = (
#     "你熟读红楼梦,西游记,水浒传,三国演义,聊斋志异,喻世明言,警世通言,醒世恒言,"
#     "初刻拍案惊奇，二刻拍案惊奇，回答尽量贴合原著，语言偏古风简洁，依据书中内容作答。"
# )


# def build_prompt(system, instruction):
#     return "<|persona|>" + system + "<|user|>" + instruction + "<|bot|>"


# #######################################################################################
# # 加载基础模型
# #######################################################################################

# def load_base_model(base_model_path, config):
#     model = GPT_MODEL(config)

#     base_sd = torch.load(base_model_path, map_location="cpu")
#     base_sd = unwrap_state_dict(base_sd)
#     base_sd = clean_state_dict_keys(base_sd)

#     model.load_state_dict(base_sd, strict=True)
#     model.to(device)
#     model.eval()

#     print("✅ 基础模型加载完成")
#     return model


# #######################################################################################
# # 加载 LoRA 模型：不改变你的模型结构
# #######################################################################################

# def load_model_with_lora(base_model_path, lora_path, config):
#     model = GPT_MODEL(config)

#     base_sd = torch.load(base_model_path, map_location="cpu")
#     base_sd = unwrap_state_dict(base_sd)
#     base_sd = clean_state_dict_keys(base_sd)

#     model.load_state_dict(base_sd, strict=True)
#     print("✅ 基础模型加载完成")

#     lora_sd = torch.load(lora_path, map_location="cpu")
#     lora_sd = unwrap_state_dict(lora_sd)
#     lora_sd = clean_state_dict_keys(lora_sd)

#     lora_keys_in_ckpt = [k for k in lora_sd.keys() if "lora" in k.lower()]

#     print("LoRA checkpoint 中 LoRA 参数数量:", len(lora_keys_in_ckpt))
#     print("LoRA checkpoint 前 5 个 key:", list(lora_sd.keys())[:5])

#     if len(lora_keys_in_ckpt) == 0:
#         raise RuntimeError("❌ LoRA 文件中没有 lora 参数")

#     old_model = model
#     ret = apply_lora_to_model(model, rank=8)

#     if ret is None:
#         print("⚠️ apply_lora_to_model 返回 None，按原地修改处理")
#         model = old_model
#     else:
#         print("✅ apply_lora_to_model 返回新模型，使用返回值")
#         model = ret

#     model_sd = model.state_dict()
#     model_lora_keys = [k for k in model_sd.keys() if "lora" in k.lower()]

#     print("模型中 LoRA 参数数量:", len(model_lora_keys))
#     print("模型中 LoRA 前 5 个 key:", model_lora_keys[:5])

#     if len(model_lora_keys) == 0:
#         raise RuntimeError(
#             "❌ apply_lora_to_model 没有把 LoRA 挂到模型上。"
#             "请检查 apply_lora_to_model 源码。"
#         )

#     model_keys = set(model_sd.keys())
#     missing_in_model = [k for k in lora_sd.keys() if k not in model_keys]

#     if missing_in_model:
#         print("❌ 以下 LoRA checkpoint key 在模型中找不到，前 20 个:")
#         for k in missing_in_model[:20]:
#             print("  ", k)

#         raise RuntimeError("❌ LoRA key 不匹配，推理结构和训练结构不一致")

#     result = model.load_state_dict(lora_sd, strict=False)

#     missing = result.missing_keys
#     unexpected = result.unexpected_keys

#     missing_lora = [k for k in missing if "lora" in k.lower()]
#     unexpected_lora = [k for k in unexpected if "lora" in k.lower()]

#     print("missing keys 总数:", len(missing))
#     print("unexpected keys 总数:", len(unexpected))
#     print("missing LoRA keys 数量:", len(missing_lora))
#     print("unexpected LoRA keys 数量:", len(unexpected_lora))

#     if missing_lora or unexpected_lora:
#         print("missing_lora 前 10 个:", missing_lora[:10])
#         print("unexpected_lora 前 10 个:", unexpected_lora[:10])
#         raise RuntimeError("❌ LoRA 权重没有完整加载")

#     print(f"✅ LoRA 微调权重挂载完成，成功加载 {len(lora_keys_in_ckpt)} 个 LoRA 参数")

#     b_stats = []
#     for n, p in model.named_parameters():
#         if "lora" in n.lower() and n.endswith("B"):
#             b_stats.append((n, p.detach().abs().sum().item()))

#     nonzero_b = sum(1 for _, v in b_stats if v > 1e-8)
#     print(f"LoRA B 非零数量: {nonzero_b}/{len(b_stats)}")

#     if len(b_stats) > 0 and nonzero_b == 0:
#         print("⚠️ 所有 LoRA B 都是 0，LoRA 可能没训练到")

#     model.to(device)
#     model.eval()

#     return model


# #######################################################################################
# # logits 兼容工具
# #######################################################################################

# def get_logits(model, input_tensor):
#     outputs = model(input_ids=input_tensor)

#     if isinstance(outputs, dict):
#         return outputs["logits"]

#     return outputs


# #######################################################################################
# # 生成函数
# #######################################################################################

# @torch.no_grad()
# def generate_text(
#     model,
#     prompt,
#     max_new_tokens=300,
#     temperature=0.7,
#     top_k=50,
#     greedy=False,
#     only_answer=True,
#     context_size=1024,
# ):
#     model.eval()

#     input_ids = encode_text(prompt)
#     input_tensor = torch.tensor([input_ids], dtype=torch.long, device=device)

#     generated_ids = []

#     for _ in range(max_new_tokens):
#         idx_cond = input_tensor[:, -context_size:]

#         logits = get_logits(model, idx_cond)[:, -1, :]

#         if greedy:
#             next_token = torch.argmax(logits, dim=-1, keepdim=True)
#         else:
#             temperature = max(float(temperature), 1e-6)
#             logits = logits / temperature

#             if top_k is not None and top_k > 0:
#                 k = min(top_k, logits.size(-1))
#                 top_logits, _ = torch.topk(logits, k)
#                 logits[logits < top_logits[:, [-1]]] = float("-inf")

#             probas = torch.softmax(logits, dim=-1)
#             next_token = torch.multinomial(probas, num_samples=1)

#         next_id = next_token.item()

#         if next_id in STOP_IDS:
#             break

#         generated_ids.append(next_id)
#         input_tensor = torch.cat([input_tensor, next_token], dim=-1)

#     if only_answer:
#         return tokenizer.decode(generated_ids)
#     else:
#         return tokenizer.decode(input_tensor[0].tolist())


# #######################################################################################
# # 诊断 1：裸测基础模型，不带 persona/user/bot
# #######################################################################################

# base_model = load_base_model(BASE_MODEL_PATH, NEW_CONFIG)

# print()
# print("=" * 60)
# print("【诊断1】基础模型纯文本续写：话说天下大势")
# print("=" * 60)

# print(generate_text(
#     base_model,
#     "话说天下大势,",
#     max_new_tokens=200,
#     greedy=True,
#     only_answer=True
# ))

# print()
# print("=" * 60)
# print("【诊断2】基础模型纯文本续写：滚滚长江东逝水")
# print("=" * 60)

# print(generate_text(
#     base_model,
#     "滚滚长江东逝水,",
#     max_new_tokens=200,
#     greedy=True,
#     only_answer=True
# ))

# #######################################################################################
# # 加载 LoRA 模型
# #######################################################################################

# model = load_model_with_lora(
#     BASE_MODEL_PATH,
#     LORA_PATH,
#     NEW_CONFIG
# )

# #######################################################################################
# # 用干净问题测试，不用脏切片问题
# #######################################################################################

# clean_instruction = "请简述时有宦官曹节等弄权，窦武、陈蕃谋诛之，机事不密这段故事的经过？"

# prompt = build_prompt(SYSTEM_TEXT, clean_instruction)

# print()
# print("=" * 60)
# print("【输入 prompt】")
# print("=" * 60)
# print(prompt)

# print()
# print("=" * 60)
# print("【测试1】LoRA + 贪心解码")
# print("=" * 60)

# print(generate_text(
#     model,
#     prompt,
#     max_new_tokens=300,
#     greedy=True,
#     only_answer=True
# ))

# print()
# print("=" * 60)
# print("【测试2】LoRA + temperature=0.3, top_k=30")
# print("=" * 60)

# print(generate_text(
#     model,
#     prompt,
#     max_new_tokens=300,
#     temperature=0.3,
#     top_k=30,
#     only_answer=True
# ))

# print()
# print("=" * 60)
# print("【测试3】LoRA + temperature=0.7, top_k=50")
# print("=" * 60)

# print(generate_text(
#     model,
#     prompt,
#     max_new_tokens=300,
#     temperature=0.7,
#     top_k=50,
#     only_answer=True
# ))

# #######################################################################################
# # 诊断 3：计算训练样本 loss
# #######################################################################################

# @torch.no_grad()
# def calc_sample_loss(model, system, instruction, output):
#     """
#     这里默认你的训练格式是:
#     <|persona|>system<|user|>instruction<|bot|>output<|endoftext|>
#     如果你训练时不是这个格式，loss 结果就没有参考意义。
#     """

#     prompt = build_prompt(system, instruction)
#     answer = output

#     if EOS_ID is not None:
#         full_text = prompt + answer + "<|endoftext|>"
#     else:
#         full_text = prompt + answer

#     ids = encode_text(full_text)
#     prompt_ids = encode_text(prompt)

#     if len(ids) < 2:
#         return None

#     x = torch.tensor([ids[:-1]], dtype=torch.long, device=device)
#     y = torch.tensor([ids[1:]], dtype=torch.long, device=device)

#     logits = get_logits(model, x)

#     vocab_size = logits.size(-1)

#     loss_all = torch.nn.functional.cross_entropy(
#         logits.reshape(-1, vocab_size),
#         y.reshape(-1),
#         reduction="mean"
#     ).item()

#     # answer 部分 loss
#     # target 的位置是原序列 ids[1:]
#     # answer 从 len(prompt_ids) 开始
#     answer_target_start = len(prompt_ids) - 1

#     if answer_target_start < y.size(1):
#         answer_logits = logits[:, answer_target_start:, :]
#         answer_y = y[:, answer_target_start:]

#         loss_answer = torch.nn.functional.cross_entropy(
#             answer_logits.reshape(-1, vocab_size),
#             answer_y.reshape(-1),
#             reduction="mean"
#         ).item()
#     else:
#         loss_answer = None

#     return {
#         "total_tokens": len(ids),
#         "prompt_tokens": len(prompt_ids),
#         "loss_all": loss_all,
#         "loss_answer": loss_answer,
#     }


# test_output = (
#     "时有宦官曹节等弄权，窦武、陈蕃谋诛之，机事不密，反为所害。"
#     "中涓自此愈横。建宁二年四月望日，帝御温德殿。"
#     "方升座，殿角狂风骤起，只见一条大青蛇从梁上飞将下来，蟠于椅上。"
#     "帝惊倒，左右急救入宫；百官俱奔避。"
# )

# print()
# print("=" * 60)
# print("【诊断3】训练格式 loss 检查")
# print("=" * 60)

# base_loss = calc_sample_loss(
#     base_model,
#     SYSTEM_TEXT,
#     clean_instruction,
#     test_output
# )

# lora_loss = calc_sample_loss(
#     model,
#     SYSTEM_TEXT,
#     clean_instruction,
#     test_output
# )

# print("基础模型 loss:", base_loss)
# print("LoRA 模型 loss:", lora_loss)


In [ ]:
# ###Qwen new train  bad bad bad bad bad
# import os, re, math, json
# from datetime import datetime
# import torch
# from torch.utils.data import Dataset, DataLoader
# from torch.optim.lr_scheduler import LambdaLR

# #######################################################################################
# # 路径 & 配置
# #######################################################################################
# ROOT = r"/kaggle/input/datasets/mangzhang/qwennewtrainclean"
# ROOTPTH = "/kaggle/input/datasets/mangzhang/allchineseliteralcn/"
# ROOTRW = "/kaggle/working/"
# CLEAN_PATH = os.path.join(ROOT, "pretrain_body_clean.txt")
# OUT_DIR = os.path.join(ROOTRW, "ckpt"); os.makedirs(OUT_DIR, exist_ok=True)

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print("device:", device)
# if device.type == "cuda":
#     print(torch.cuda.get_device_name(0),
#           f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# CTX     = min(1024, NEW_CONFIG["context_length"])
# STRIDE  = 512        # 50% overlap → 样本翻倍
# BS      = 4
# ACCUM   = 8          # 等效 batch 32
# EPOCHS  = 12
# LR      = 3e-4       # ★ 关键:从 1e-5 提到 3e-4
# WARMUP  = 400
# WD      = 0.1
# SAVE_EVERY = 500
# USE_AMP = (device.type == "cuda")

# #######################################################################################
# # 读语料 + 最后清理
# #######################################################################################
# text = open(CLEAN_PATH, encoding="utf-8").read()
# text = re.sub(r"[，,]?[^。\n]{0,15}著介绍[：:]", "", text)
# text = re.sub(r"^[，。：:、；]+", "", text).strip()

# n_tok_all = len(tokenizer.encode(text))
# print(f"字数 {len(text):,} | token {n_tok_all:,} | 比值 {n_tok_all/len(text):.2f}")

# #######################################################################################
# # Dataset
# #######################################################################################
# class PretrainDataset(Dataset):
#     def __init__(self, text, ctx=CTX, stride=STRIDE):
#         ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
#         self.chunks = [torch.tensor(ids[i:i+ctx+1], dtype=torch.long)
#                        for i in range(0, len(ids)-ctx-1, stride)]
#         print(f"样本数 {len(self.chunks):,} (ctx={ctx}, stride={stride})")
#     def __len__(self): return len(self.chunks)
#     def __getitem__(self, i):
#         c = self.chunks[i]
#         return {"input_ids": c[:-1], "labels": c[1:],
#                 "attention_mask": torch.ones_like(c[:-1])}

# def collate(b): return {k: torch.stack([x[k] for x in b]) for k in b[0]}

# ds = PretrainDataset(text)
# dl = DataLoader(ds, batch_size=BS, shuffle=True, collate_fn=collate,
#                 drop_last=True, num_workers=0)   # Windows 建议 0

# steps_per_epoch = len(dl) // ACCUM
# total_steps = EPOCHS * steps_per_epoch
# print(f"每 epoch opt-step {steps_per_epoch} | 总 opt-step {total_steps}")
# print(f"预计见 token 总量 ≈ {total_steps*BS*ACCUM*CTX/1e6:.0f} M")

# #######################################################################################
# # scheduler / checkpoint
# #######################################################################################
# def sched_fn(step):
#     if step < WARMUP: return (step+1)/WARMUP
#     p = min(1.0, (step-WARMUP)/max(1, total_steps-WARMUP))
#     return 0.1 + 0.9*0.5*(1+math.cos(math.pi*p))

# def save_ckpt(model, opt, sch, ep, gs, hist, tag="last"):
#     torch.save({"epoch":ep, "global_step":gs,
#                 "model_state_dict":model.state_dict(),
#                 "optimizer_state_dict":opt.state_dict(),
#                 "scheduler_state_dict":sch.state_dict(),
#                 "config":NEW_CONFIG, "history":hist},
#                os.path.join(OUT_DIR, f"pretrain_ckpt_{tag}.pth"))
#     torch.save(model.state_dict(),
#                os.path.join(OUT_DIR, f"allChineseLiteralCN_{tag}.pth"))
#     print(f"💾 {tag} @ step {gs}")

# @torch.no_grad()
# def probe(model, n=100):
#     model.eval()
#     for p in ["话说天下大势，", "滚滚长江东逝水，", "冤屈难伸，只见"]:
#         print(f"  {p} → {generate_text(model, p, max_new_tokens=n, greedy=True)}")
#     model.train()

# #######################################################################################
# # 训练
# #######################################################################################
# def train(resume=None, load_old_base=True):
#     model = GPT_MODEL(NEW_CONFIG)
#     if load_old_base:
#         p = os.path.join(ROOTPTH, "allChineseLiteralCN.pth")
#         model.load_state_dict(torch.load(p, map_location="cpu"), strict=True)
#         print("✅ 载入旧 base 继续训")

#     # 数据量有限 → 关 dropout,利于 memorize
#     model.drop_emb.p = 0.0
#     for blk in model.trf_blocks:
#         for m in blk.modules():
#             if isinstance(m, torch.nn.Dropout): m.p = 0.0
#     print("✅ dropout 已置 0")

#     model.to(device)
#     opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD, betas=(0.9,0.95))
#     sch = LambdaLR(opt, sched_fn)
#     scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

#     ep0, gs, hist = 0, 0, []
#     if resume and os.path.exists(resume):
#         ck = torch.load(resume, map_location="cpu")
#         model.load_state_dict(ck["model_state_dict"])
#         opt.load_state_dict(ck["optimizer_state_dict"])
#         sch.load_state_dict(ck["scheduler_state_dict"])
#         ep0, gs, hist = ck["epoch"], ck["global_step"], ck.get("history", [])
#         print(f"▶️ resume ep{ep0} step{gs}")

#     t0 = datetime.now()
#     for ep in range(ep0, EPOCHS):
#         model.train(); tot, cnt = 0., 0
#         opt.zero_grad(set_to_none=True)

#         for i, b in enumerate(dl):
#             with torch.cuda.amp.autocast(enabled=USE_AMP, dtype=torch.float16):
#                 out = model(input_ids=b["input_ids"].to(device),
#                             attention_mask=b["attention_mask"].to(device),
#                             labels=b["labels"].to(device))
#                 loss = out["loss"] / ACCUM

#             scaler.scale(loss).backward()

#             if (i+1) % ACCUM == 0:
#                 scaler.unscale_(opt)
#                 torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
#                 scaler.step(opt); scaler.update()
#                 sch.step(); opt.zero_grad(set_to_none=True)
#                 gs += 1

#                 l = loss.item()*ACCUM
#                 hist.append(l)
#                 if gs % 20 == 0:
#                     el = (datetime.now()-t0).total_seconds()/60
#                     print(f"[{el:6.1f}min] ep{ep} gs{gs}/{total_steps} "
#                           f"loss {l:.4f} lr {sch.get_last_lr()[0]:.2e}")
#                 if gs % SAVE_EVERY == 0:
#                     save_ckpt(model,opt,sch,ep,gs,hist,"last")
#                     probe(model)

#             tot += loss.item()*ACCUM; cnt += 1

#         print(f"===== epoch {ep} avg loss {tot/cnt:.4f} =====")
#         save_ckpt(model,opt,sch,ep+1,gs,hist,f"ep{ep}")
#         probe(model, n=150)

#     save_ckpt(model,opt,sch,EPOCHS,gs,hist,"final")
#     return model, hist

# base_model, hist = train(load_old_base=True)


In [ ]:
# def download_and_load_file(file_path): 
#     with open(file_path, "r", encoding="utf-8") as file: 
#             text_data = file.read() 
#     with open(file_path, "r") as file: 
#         data = json.load(file) 
#     return data 
# file_path = "/kaggle/input/datasets/mangzhang/sf-data/sft_data.json" 

# data = download_and_load_file(file_path) 
# print("Number of entries:", len(data))

In [ ]:
# def format_input(entry): 
#     instruction_text = ( 
#         f"Below is an instruction that describes a task. " 
#         f"Write a response that appropriately completes the request." 
#         f"\n\n### Instruction:\n{entry['instruction']}" 
#     ) 
#     input_text = ( 
#         f"\n\n### Input:\n{entry['input']}" if entry["input"] else "" 
#     ) 
#     return instruction_text + input_text

In [ ]:
# model_input = format_input(data[999]) 
# desired_response = f"\n\n### Response:\n{data[999]['output']}" 
# print(model_input + desired_response)

In [ ]:
# train_portion = int(len(data)*0.85)
# test_portion = int(len(data)*0.1)
# val_portion = len(data) - train_portion - test_portion
# train_data = data[:train_portion]
# test_data = data[train_portion:train_portion+test_portion]
# val_data = data[train_portion+test_portion:]
# print("Training set length:", len(train_data)) 
# print("Validation set length:", len(val_data)) 
# print("Test set length:", len(test_data))

In [ ]:
# class InstructionDataset(Dataset):
#     def __init__(self, data, tokenizer):
#         self.data = data
#         self.encoded_texts = []
#         for entry in data:
#             instruction_plus_input = format_input(entry)
#             response_text = f"\n\n### Response: \n{entry["output"]}"
#             full_text = instruction_plus_input + response_text
#             self.encoded_texts.append(
#                 tokenizer.encode(full_text)
#             )
#     def __getitem__(self, index):
#          return self.encoded_texts[index]
#     def __len__(self):
#          return len(self.data)

In [ ]:

# inputs_1 = [0, 1, 2, 3, 4] 
# inputs_2 = [5, 6] 
# inputs_3 = [7, 8, 9] 
# batch = ( 
#     inputs_1, 
#     inputs_2, 
#     inputs_3 
# ) 
# def custom_collate_fn(
#     batch,
#     pad_token_id=50259,
#     ignore_index=-100,
#     allowed_max_length=None,
#     device="cpu"
# ):
#     batch_max_length = max(len(item) +1 for item in batch)
#     inputs_lst, targets_lst = [], []
#     for item in batch:
#         new_item = item.copy()
#         new_item += [pad_token_id]
#         padded = (
#             new_item + [pad_token_id] *
#             (batch_max_length - len(new_item))
#         )
#         inputs = torch.tensor(padded[:-1])
#         targets = torch.tensor(padded[1:])
#         mask = targets == pad_token_id
#         indices = torch.nonzero(mask).squeeze()
#         if indices.numel() > 1:
#             targets[indices[1:]] = ignore_index
#         if allowed_max_length is not None:
#             inputs = inputs[:allowed_max_length]
#             targets = targets[:allowed_max_length]
            
#         inputs_lst.append(inputs)
#         targets_lst.append(targets)
    
#     inputs_tensor = torch.stack(inputs_lst).to(device)
#     targets_tensor = torch.stack(targets_lst).to(device)
    
#     return inputs_tensor, targets_tensor
# inputs, targets = custom_collate_fn(batch)
# print(inputs)
# print(targets)

In [ ]:
# from functools import partial
# customized_collate_fn = partial (
#     custom_collate_fn,
#     device=device,
#     allowed_max_length=1024
# )

In [ ]:
# num_workers =0
# batch_size=8
# torch.manual_seed(123)
# train_dataset = InstructionDataset(train_data, tokenizer)
# train_loader = DataLoader(
#     train_dataset,
#     batch_size=batch_size,
#     collate_fn=customized_collate_fn,
#     shuffle=True,
#     drop_last=True,
#     num_workers=num_workers
# )
# val_dataset = InstructionDataset(val_data, tokenizer)
# val_loader = DataLoader(
#     val_dataset,
#     batch_size=batch_size,
#     collate_fn=customized_collate_fn,
#     shuffle=False,
#     drop_last=False,
#     num_workers=num_workers
# )
# test_dataset = InstructionDataset(test_data, tokenizer)
# test_loader = DataLoader(
#     test_dataset,
#     batch_size=batch_size,
#     collate_fn=customized_collate_fn,
#     shuffle=False,
#     drop_last=False,
#     num_workers=num_workers
# )
# print("Train loader:") 
# for inputs, targets in train_loader: 
#     print(inputs.shape, targets.shape)

In [ ]:
# from gpt_download import download_and_load_gpt2 

# CHOOSE_MODEL = "gpt2-medium (124M)" 
# NEW_CONFIG.update(model_configs[CHOOSE_MODEL]) 
# model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")") 
# settings, params = download_and_load_gpt2( 
#     model_size=model_size, 
#     models_dir="gpt2" 
# ) 
# model = GPTModel(NEW_CONFIG) 
# load_weights_into_gpt(model, params) 
# model.eval();